# Basketball

In [ ]:
#| include: false

import warnings
warnings.filterwarnings('ignore')

"It was the first time someone was speaking my language in basketball...It wasn't just [about] heart and grit." -- Former NBA Player Shane Battier upon joining the Houston Rockets in 2006 and working with analytics-minded general manager Daryl Morey.

"When I was on the Rockets, Hakeem would tell me, 'Play simple.' And I would tell him 'I am, all I do is give you the ball!' He said, 'Yes, and when I get the ball, I either score, get fouled, or you get an open three.' That is analytics and they have been around forever." -- Scott Brooks sharing an anecdote at the 2016 Sloan Sports Analytics Conference about playing with Hakeem Olajuwon.

"Everybody pulls for David, nobody roots for Goliath." -- Wilt Chamberlain

## Python Modules

We will use the following Python modules in this chapter that we have used in previous chapters:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

We will also use the following modules in this chapter:

In [ ]:
import isapy_data
import time
from scipy import stats
from scipy.stats import norm
from statsmodels.distributions.empirical_distribution import ECDF
from nba_api.stats.endpoints import shotchartdetail
from sportypy.surfaces.basketball import NBACourt, WNBACourt

If you do not remember how to install new modules in Python, refer to Chapter 2 where there is a detailed discussion.

The textbook's `isapy_data` module contains three data sets that are used extensively throughout this chapter: `nba_games_2025`, `nba_adv_team_2025`, and `nba_ff_team_2025`. The first data set contains the game log information for each regular season game in the 2024 – 2025 NBA season. The latter two are the advanced and four factor team statistics for each of the 2024 – 2025 regular season games, respectively. The module also contains `nba_nuggets_shots`, which has every Denver Nuggets shot from the 2014/15 through 2022/23 seasons. Additionally, the module includes WNBA shot data: `wnba_seattle_pbp_2025` contains play-by-play shot data for all Seattle Storm games in the 2025 season, and `wnba_player_shots_2025` has shot locations for Breanna Stewart and A'ja Wilson during the 2025 WNBA season.

We will also be using multiple regression as an analysis technique. If you do not remember that from previous chapters, see the more complete description in the baseball chapter.

In [ ]:
games = isapy_data.load_data("nba_games_2025")
adv = isapy_data.load_data("nba_adv_team_2025")
ff = isapy_data.load_data("nba_ff_team_2025")

## If You Don't Know Anything About Basketball

Basketball is a simple enough game that older adults and young children can play it using exactly the same parameters as those who play it at the highest levels. The court is still 94 feet long, the basket is still 10 feet in the air, and the ball is the same size, albeit women's competitions use a 28.5-inch ball instead of the 29.5-inch ball that the men use. The real difference, of course, is in the skill of the players.

At it's simplest form, two teams of 5 players each try to score more points than the other team over either two halves or four quarters. In the NBA, the game is played through four 12-minute quarters. Points are scored by throwing the ball (shooting) into the basket that is perched generally 10 feet (3.05 meters) over the floor, one at each end of the court representing the goal for each team. Two points are awarded for a made basket, and if the shot was taken from a long way away from the basket, behind an arc approximately 23 feet (7 meters) from the basket, three points are awarded. The defense tries to disrupt the players from making the baskets by standing close to them and trying to take the ball away (steal) from them or knock the ball away as it leaves the shooter's hands (block). If the defense were to knock the player's arms or legs or push them while they are shooting, a foul is called on the defender and the shooter will get two attempts to make an unimpeded shot (free throws) from a line 15 feet (4.57 meters) directly in front of the basket. Each free throw is worth one point. If a player commits five or six fouls (depending on the league) during the course of a game, that player is disqualified for the rest of the game. When the game is played on the playground, it is often played as the first team to reach a certain number of baskets (usually 15) with shots behind the 3-point line counting double.

There are of course many more nuances to the game, but this gives you the basics of how to play basketball. You can see much more information at [https://www.rookieroad.com/basketball/](https://www.rookieroad.com/basketball/) on the play of the game and strategies at different levels.

Basketball has grown to perhaps the second most popular sport globally, behind only soccer. While it is the one major team sport played in America that is likely uniquely and authentically American, there have been many efforts to grow the game around the world. The USA still dominates the men's Olympic basketball tournament, having won the gold medal in 16 of the 19 Olympics in which it has participated, but hotbeds of basketball are developing around the world. China, Western Europe, Australia, the Baltic Republics, and the Adriatic countries are large markets for basketball and send many players to play in professional and amateur leagues in the United States.

For the purposes of this book, we will be using the dimensions of an NBA court and games played under NBA rules. The international court is slightly smaller and the 3-point line is also slightly closer under international rules, and there are other more subtle differences in international rules, but NBA players usually have no trouble adjusting when going back and forth between NBA games and international competitions such as the Olympic Games. FIBA, the governing body for basketball globally, has continually made the court and the rules more similar to NBA rules over the years.

![Diagram of an NBA Basketball court.](fig/bball-court.jpeg){#fig-nba-court}

## Basic Basketball Metrics

Unlike baseball which we examined in the prior chapter, basketball is free-flowing and is not a collection of discrete events that can be studied individually or in the whole. That is, there are not defined breaks between pitches or balls in play in basketball. It can be divided somewhat accurately into possessions, and certainly we can track things like shots and rebounds (though both of those with a bit of a margin for error when balls are being tipped around), fouls, time played by each player, and the like. With this in mind, we will apply analytics a bit differently than we do in ways like baseball.

| **Offensive**                 | **Defensive**                            |
|-------------------------------|------------------------------------------|
| Points Scored                 | Minutes played: Amount of game time a player is one of the five people active on the court. |
| Field Goal Percentage: Number of shots made/number of shots attempted  | Defensive Rebounds: Catching the ball as it bounces off of the rim after an opponent's missed shot (Offensive Rebounds, when the rebounder is on the same team as the shooter, could be considered an offensive metric.)  |
| Free Throw Percentage: Number of free throws made/number of free throws attempted | Blocks: Swatting the ball away as it leaves the opponent's hand |
| Assists: Passes that directly result in a teammate scoring a field goal | Steals: Taking the ball away from a player without committing a foul. |
| Three-Point Shots: Shots made from behind the three-point line | Fouls: breaking the rules of permissible conduct on the floor as it relates to interacting with an opponent. Usually this is physical contact with a player in the act of shooting. |
| Turnovers: Committing an act (other than a foul or a missed shot) that causes the other team to gain possession of the ball. This can often be throwing the ball out of bounds on a bad pass. | Plus/Minus: The number of points your team scored while you were in the game less the number of points your team allowed while you were in the game. |

: Early metrics used in basketball analytics. {#tbl-early-basketball}

## Four Factors

The notion of the "four factors" was developed by Dean Oliver and is explained in his book Basketball on Paper, available at [http://basketballonpaper.com](http://basketballonpaper.com). His original formulation assigned the following weight to wins: Shooting (40%), Turnovers (25%), Rebounding (20%), and Free Throws (15%). There has been a lot of debate about the appropriateness of the weighting of these factors, and we will look at some aspects of weighting in this chapter. Most people still agree, however, that the four factors are extremely important to a team's success. Note, too, that while these four factors describe a team's offensive efficiency, they can also be used to describe the efficiency of a team's defense (i.e., what is our opponent's effective field goal percentage?)

1. Effective field goal percentage
2. Turnovers committed per possession
3. Offensive rebounding percentage
4. Free throw rate

### Effective Field Goal Percentage {.unnumbered}

Effective Field Goal Percentage (EFG) is a means of resolving the difference between two point and three-point field goals. It gives more weight to three pointers relative to two pointers. How much more weight? It adds 50% more to a three pointer because they are worth 50% more points. The exact formula is

$$EFG = \frac{fg2m + 1.5 \times fg3m}{fga}.$$

where fg2m and fg3m are the number of two point and three point baskets made, respectively, and fga is the number of overall shots attempted.

Upon examining the `nba_games_2025` data frame (which you will find in the `isapy_data` module), we note that we can create a new column to calculate EFG, or we could also just look at the advanced box score for each game. Let's examine these and compare the figures from both tables.

In [ ]:
games['fg2m'] = games['fgm'] - games['fg3m']
games['efg'] = (games['fg2m'] + 1.5 * games['fg3m']) / games['fga']

In [ ]:
#| label: fig-efgp-calcs
#| fig-cap: "The effective field goal percentage from our calculations (x) versus what is reported on NBA.com (y)."

df_merge = games.merge(adv, on=['team_id', 'team_abbreviation', 'game_id', 'min'],
                       suffixes=('', '_adv'))

plt.figure(figsize=(8, 6))
plt.scatter(df_merge['efg'], df_merge['efg_pct'], alpha=0.5)
plt.xlabel('EFG (Our Calculation)')
plt.ylabel('EFG (From NBA.com)')
plt.title("Comparing our calculated EFG to NBA.com's EFG")
plt.tight_layout()
plt.show()

We find that these are the same in both sets, which is good news for our calculations!

### Turnovers Committed per Possession (TPP) {.unnumbered}

As the name suggests, TPP is the number of turnovers committed relative to the number of possessions the team had during the game. Turnovers (`tov`) and possessions (`poss`) are given in the `nba_games_2025` and `nba_adv_team_2025` DataFrames, respectively.

In [ ]:
df_merge['tpp'] = df_merge['tov'] / df_merge['poss']

### Offensive Rebounding Percentage (ORP) {.unnumbered}

Roughly, ORP is the ratio of the total number of offensive rebounds to the number of offensive rebound possibilities. Note that both offensive and defensive rebounds are given in the `nba_games_2025` DataFrame and the actual rates are given in the `nba_ff_team_2025` and `nba_adv_team_2025` DataFrames. To calculate a team's ORP for a given game, offensive rebound percentage is simply

$$
ORP = \frac{\textrm{team offensive rebounds}}{\textrm{team offensive rebounds} + \textrm{opponent defensive rebounds}}.
$$

### Free Throw Rate (FTR) {.unnumbered}

FTR is the ratio of free throw attempts (`fta`) to field goal attempts (`fga`). Both statistics are given in the `nba_games_2025` DataFrame and FTR is given in `nba_ff_team_2025`. Let's look below at manually calculating the FTR in the `nba_games_2025` DataFrame and compare it to what is reported in the `nba_ff_team_2025` table. We will print the first few rows as a proof of concept.

In [ ]:
ftr_check = games.merge(ff, on=['team_id', 'team_abbreviation', 'game_id', 'min'],
                        suffixes=('', '_ff'))
ftr_check['ftr'] = ftr_check['fta'] / ftr_check['fga']
ftr_check[['fta_rate', 'ftr']].head()

These four factors are Oliver's representations of the areas where teams need to excel. Above, we showed how to obtain the data to calculate these metrics. We will continue to explore the four factors throughout the rest of the chapter.

## Densities, Percentiles, and Z Scores

In the last section we saw how to calculate and retrieve values for the four factors, but we did not discuss their usefulness. So how can we use the four factors? We could examine the distribution of values for a single (or multiple) season(s). Let's first look at these values for each team in the 2024 – 2025 NBA season. We will do this with a density plot, which shows us the distribution of EFG across all teams in this season.

### Density {.unnumbered}

In [ ]:
#| label: fig-efgp-density
#| fig-cap: "A density plot of effective field goal percentage."

plt.figure(figsize=(8, 6))
sns.kdeplot(data=adv, x='efg_pct')
plt.xlabel('Effective Field Goal Percentage')
plt.title('2025 NBA EFG')
plt.tight_layout()
plt.show()

We can see that the highest density of team EFG is somewhere between 50-55%, but what is the average in this sample and can we add a reference line to this previous plot to represent it? Certainly we can:

* We will use `.mean()` to calculate the mean, and then
* We will use `ax.axvline()` to add the vertical line
    + we will set the x position of that line to the value of the mean
    + we will make the line dashed so that it stands out from the gridlines.

In [ ]:
adv['efg_pct'].mean()

In [ ]:
#| label: fig-efgp-dens-mean
#| fig-cap: "A density plot of effective field goal percentage along with the mean reference line."

plt.figure(figsize=(8, 6))
sns.kdeplot(data=adv, x='efg_pct')
plt.axvline(x=adv['efg_pct'].mean(), linestyle='--', color='black')
plt.xlabel('Effective Field Goal Percentage')
plt.tight_layout()
plt.show()

One way to view the graphical results is to plot the four factors all at the same time. We can do this by creating a faceted plot. To do this, we need to reshape our data from wide to long format using `pd.melt()`. First, we merge our previous data set with the four factors data (`nba_ff_team_2025`). Next we define a `win` variable to `True` if the game was a win, calculated by if the variable `plus_minus` is greater than 0 (meaning that the team scored more points than they allowed), and otherwise set it to `False`, meaning the game was a loss. Finally, we reformat the data set to a long format from the traditional wide format. The long format denormalizes the data so that we can print the plots as facets. Let's look at the entire dataset visually first before getting into wins and losses.

In [ ]:
df_pivot = df_merge.merge(ff, on=['team_id', 'team_abbreviation', 'game_id', 'min'],
                          suffixes=('', '_ff'))
df_pivot['win'] = df_pivot['plus_minus'] > 0
df_pivot['tov_pct'] = df_pivot['tm_tov_pct'] / 100

df_four = df_pivot[['efg_pct', 'oreb_pct_ff', 'tov_pct', 'fta_rate', 'win']].copy()
df_four.columns = ['efg_pct', 'oreb_pct', 'tov_pct', 'fta_rate', 'win']

df_long = df_four.melt(id_vars=['win'], var_name='factor', value_name='value')

In [ ]:
#| label: fig-ff-dens
#| fig-cap: "Density plots of the four factors for the 2025 season."

g = sns.FacetGrid(df_long, col='factor', col_wrap=2, sharex=False, height=4)
g.map(sns.kdeplot, 'value')
g.set_axis_labels('Value', 'Density')
g.fig.suptitle('NBA Four Factors (2025 Season)', y=1.02)
plt.tight_layout()
plt.show()

Now, let's see how the results of the four factors compare in games that were wins and games that were losses. We can do this now by changing the aesthetics to plot the values of each of the four factors on the x-axis by each of the values for the win variable (`True` and `False` as we calculated above).

In [ ]:
#| label: fig-ff-dens-wl
#| fig-cap: "Density plots of the four factors by wins and losses for the 2025 season."

g = sns.FacetGrid(df_long, col='factor', col_wrap=2, hue='win',
                  sharex=False, height=4, palette='Set1')
g.map(sns.kdeplot, 'value', fill=True, alpha=0.25)
g.add_legend(title='Win')
plt.tight_layout()
plt.show()

Mostly it seems that there is a lot of overlap, with the biggest difference being in EFG. EFG being different makes sense on its face, as games are graded not on rebounds or turnovers but rather on points, and higher EFG should yield higher points scored, which should create more wins. But why do you think there is not a large difference in the other statistics?

## Specific Game Evaluation

![](fig/la-den.png)

In 2023, the Denver Nuggets won the NBA Championship. How did they fare against the Los Angeles Lakers in the 2024-25 regular season? We can turn our attention to the Denver Nuggets @ Los Angeles Lakers game on November 23, 2024. The Nuggets won this game 127 to 102, but the question remains "did they play a good game?" Similarly, did the Lakers necessarily play a bad game given that they lost? One could simply point to the scoreboard and say that this is all that is needed to evaluate the results, however, given that we are interested in sports analytics and seek to remove the anomalous results from our overall evaluation, we can do deeper dives into the data. To do this, we need to filter for the appropriate game in the `nba_games_2025` table, get the game id, and then use that to filter the advanced statistics table.

In [ ]:
den_lal_id = games[
    (games['game_date'] == '2024-11-23') &
    (games['team_abbreviation'].isin(['LAL', 'DEN']))
]['game_id'].iloc[0]

den_lal_adv = adv[adv['game_id'] == den_lal_id]

Now let's create some lines to show how both the Nuggets and the Lakers performed on this night with respect to EFG. This will show the performances for this one game as opposed to all the games from the 2025 season. How would you interpret these results?

In [ ]:
#| label: fig-efgp-dens-den-lal
#| fig-cap: "Effective field goal percentage density plot from the 2025 season along with the specific values from the Denver Nuggets @ Los Angeles Lakers game on November 23, 2024."

plt.figure(figsize=(8, 6))
sns.kdeplot(data=adv, x='efg_pct')

for _, row in den_lal_adv.iterrows():
    color = '#4D90CD' if row['team_abbreviation'] == 'DEN' else '#FDB927'
    plt.axvline(x=row['efg_pct'], color=color, label=row['team_abbreviation'])

plt.legend(title='Team')
plt.xlabel('Effective Field Goal Percentage')
plt.title('Nuggets @ Lakers 11/23/2024 EFG vs 2024-25 NBA Averages')
plt.tight_layout()
plt.show()

It seems that the Lakers had a below average night and the Nuggets were significantly better than average, relative to EFG. Let's take the same logic and apply it to ORP. What is your interpretation of this plot?

In [ ]:
#| label: fig-oreb-dens-den-lal
#| fig-cap: "Offensive rebounding percentage density plot from the 2025 season along with the specific values from the Denver Nuggets @ Los Angeles Lakers game on November 23, 2024."

plt.figure(figsize=(8, 6))
sns.kdeplot(data=adv, x='oreb_pct')

for _, row in den_lal_adv.iterrows():
    color = '#4D90CD' if row['team_abbreviation'] == 'DEN' else '#FDB927'
    plt.axvline(x=row['oreb_pct'], color=color, label=row['team_abbreviation'])

plt.legend(title='Team')
plt.xlabel('Offensive Rebound Percentage')
plt.title('Nuggets @ Lakers 11/23/2024 OREB vs 2024-25 NBA Averages')
plt.tight_layout()
plt.show()

## Computing Estimated Probabilities

We still need to answer the question of whether the teams played particularly well or not on this night. To do this, we can look at the EFG percentage that we calculated above and compare it to the probability of the events happening. We will use the `ECDF()` function from statsmodels to find where in the cumulative distribution that the points for Denver's and Los Angeles' EFG appear given all the games in the 2024-25 season.

In [ ]:
ecdf_fn = ECDF(adv['efg_pct'].dropna())
for _, row in den_lal_adv.iterrows():
    pctile = ecdf_fn(row['efg_pct'])
    print(f"{row['team_abbreviation']}: {pctile:.4f}")

This is saying that the percentages shown above of the teams in this data set (all 2025 games) had Effective Field Goal Percentages less than the respective team in this game.

## Z-Scores

You may also remember Z-scores from statistics as a calculation of the number of standard deviations that an observation is from the mean. We can calculate the Z-score for the Offensive Rebounding percentages that we examined earlier in the chapter. Recall the formula for a Z-score is

$$
z = \frac{\textrm{observed value - mean}}{\text{standard deviation}}.
$$

The code below shows the Z-scores for Los Angeles and Denver in terms of offensive rebounding percentage.

In [ ]:
z_oreb = (den_lal_adv['oreb_pct'].values - adv['oreb_pct'].mean()) / adv['oreb_pct'].std()
for team, z in zip(den_lal_adv['team_abbreviation'], z_oreb):
    print(f"{team} OREB Z-score: {z:.4f}")

We can compute percentiles assuming a normal distribution by using the `norm.cdf()` function from `scipy.stats`. We use the Z-scores computed above as the input into the `norm.cdf()` function.

In [ ]:
percentiles = norm.cdf(z_oreb)
for team, p in zip(den_lal_adv['team_abbreviation'], percentiles):
    print(f"{team} Normal percentile: {p:.4f}")

## Using the Four Factors to Predict Wins and Losses

Another way we could use the four factors is to predict their effects on whether or not a team wins a game. To do this, we will use logistic regression. We touched on this in the chapter on baseball, and we will use it again here.

In this case, we will use the logit model with win as the dependent variable. We define a new DataFrame, `df_reg`, by merging the game data with the four factors data. Note that we must remove the variable `plus_minus`, because win is directly derived from `plus_minus`. To fit the model, we write that the logit of `win` is a linear function of the four factors. We also tell the logit function that the dependent variable is binary as it is either a win or a loss. We then print the output.

In [ ]:
df_reg = df_merge.merge(ff, on=['team_id', 'team_abbreviation', 'game_id', 'min'],
                        suffixes=('', '_ff'))
df_reg['win'] = (df_reg['plus_minus'] > 0).astype(int)
df_reg['tov_pct'] = df_reg['tm_tov_pct'] / 100

df_reg_model = df_reg[['efg_pct', 'oreb_pct_ff', 'tov_pct', 'fta_rate', 'win']].copy()
df_reg_model.columns = ['efg_pct', 'oreb_pct', 'tov_pct', 'fta_rate', 'win']
df_reg_model = df_reg_model.dropna()

adv_glm = smf.logit('win ~ efg_pct + oreb_pct + tov_pct + fta_rate',
                      data=df_reg_model).fit(disp=0)
print(adv_glm.summary())

You may look at this output and not remember what all this means, or maybe in the baseball chapter was the first time you saw it. Or both. As we discussed there, the important things to note on this is the p-value column (labeled `P>|z|`). P-values are the probability that the component is contributing to the model purely by chance and not because it has real predictive power. Generally speaking, whenever the p-value is less than .05, we want to consider that variable as an important predictor in our model.

Also important with the coefficients is their sign. EFG, ORP, and FTA all have positive signs, which means that as their value increases, the probability of a win will also increase. Conversely, the sign of the TPP coefficient is negative and, hence, the probability of a win will decrease as TPP increases. This should make sense as we would expect high levels of EFG, ORP, and FTA to correlate with wins, and high levels of TPP to correlate with losses.

An alternative formulation of the model specification given above is to explicitly write out the independent variables that are contained in the model, as we have done above.

In [ ]:
#| eval: false

adv_glm = smf.logit('win ~ efg_pct + oreb_pct + tov_pct + fta_rate',
                      data=df_reg_model).fit(disp=0)

___

## Research Block: Is the season over after 20 games in the NBA?

In the United States, Thanksgiving Day is a holiday celebrated on the fourth Thursday in November. While known as a time for giving thanks and celebrating with family and friends over a large dinner feast, it is also synonymous with NFL football on TV. It is similar to Boxing Day cricket in Australia, or with the NBA, Christmas Day. However, Thanksgiving is also a day on the calendar which marks approximately the 20 games played mark for NBA teams. This is only about one fourth of the way through an 82-game season, but yet is important for other reasons, as it is often viewed as a predictor of which 20 of 30 teams will make the playoffs (and which 8 of those teams will have to play an additional round of games).

Author Ryan Elmore in his paper "Predicting which teams will make the NBA playoffs" published in the Italian Journal of Applied Statistics, available at [https://doi.org/10.26398/IJAS.0030-009](https://doi.org/10.26398/IJAS.0030-009) considered data from the 2002 – 2015 NBA seasons to investigate this problem. Using factors such as did the team make the prior year's playoffs, the number of away games, among others, they fit a series of generalized linear models to predict if a team made the playoffs in their current season based on the covariates at any point in the season. The research suggests that you can predict the teams that will make the playoffs with about 80% accuracy after only 20 games.

So to answer the question "Are you who you are (overall) after 20 games?", the answer appears to be yes. So what does that matter for the teams involved? Teams can then position themselves perhaps for the future, through player acquisition or salary dumping, preparing for future draft selections, or deciding whether or not to try to win this year or in future years. Yet as they make these decisions, each team still has approximately 31 home games to sell for the current year. Current decisions can affect media coverage, ticket and merchandise sales, as well as the overall perception of the franchise. Even potential players may have their decisions about whether or not to join a team based on whether or not this team is viewed as a win maximizer or a profit maximizer.

___

## Shot Charts

While we have in the previous section examined the four factors and calculated their effectiveness in predicting wins and losses, one area we have left unexplored is individual performance. EFG considers all two-point shots to be equivalent, no matter where on the court they happen between the three-point arc and the basket, and it also considers all three-point shots to be equivalent no matter where beyond the arc they occur. We can use additional visualizations and analysis to create shot charts for players or teams to show greater detail on shot selection and success rates.

Coaches use shot charts to identify where players are shooting the ball, and where they are making shots. Basketball analytics has changed much about shot selection for players and teams. For example, a layup from a foot away from the basket counts for the same two points as a jump shot from 19 feet away from the hoop, and the success rate on the layup is much greater than that of the mid-range jumper. Moreover, the power of the 3-point shot has been noted as coaches look at points per possession and recognize that a 35% 3 point success rate is better in points per possession (1.05) than a 50% 2 point shot success rate (1.00). So if players are going to shoot the ball inside the 3 point arc, they should probably work to get a shot from as close to the basket as possible. Similarly, not all 3-point shots are created equally, as the shot from 25 feet counts the same as one from 75 feet away. All-star players like Steph Curry and Damian Lillard are uncommonly accurate from 30 feet or more away from the basket, and neither are afraid to shoot from that distance. Being accurate from longer range forces opposing coaches to stretch out the defenses further away from the basket, which creates more opportunities for passing or dribbling all the way to the hoop unmolested for an easy layup attempt.

### Drawing the Court {.unnumbered}

In order to construct NBA shot charts, we first need a way to draw a basketball court. We use the `sportypy` module, which provides accurate court drawings for a variety of sports including the NBA. The `NBACourt` class draws an NBA half court on a matplotlib axes with customizable colors and display options.

The shot data from the NBA Stats API uses coordinates in tenths of a foot, with the origin at the basket (`LOC_X` = 0, `LOC_Y` = 0). We need to transform these coordinates to match sportypy's coordinate system. When using `display_range='offense'`, the basket is located at approximately $x = 42$, $y = 0$, and the half court line is at $x = 0$. The coordinate transformation is:

$$x_{\text{court}} = \frac{420 - \text{LOC\_Y}}{10}, \quad y_{\text{court}} = \frac{\text{LOC\_X}}{10}.$$

### Fetching Shot Data {.unnumbered}

We will use the `nba_api` module to fetch shot chart data from the NBA Stats API. The following code will download every shot by Nikola Jokić during the 2024-25 season.

In [ ]:
jokic_id = 203999
den_team_id = 1610612743

time.sleep(1)
shot_chart = shotchartdetail.ShotChartDetail(
    team_id=den_team_id,
    player_id=jokic_id,
    season_nullable='2024-25',
    season_type_all_star='Regular Season'
)
jokic_shots = shot_chart.get_data_frames()[0]
print(f"Total shots by Jokić in 2024-25: {len(jokic_shots)}")

In [ ]:
#| label: fig-jokic-25
#| fig-cap: "Shot chart for the Denver Nuggets' center, Nikola Jokić."

fig, ax = plt.subplots(figsize=(10, 9))
court = NBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})
court.draw(ax=ax, display_range='offense')
ax.scatter((420 - jokic_shots['LOC_Y']) / 10, jokic_shots['LOC_X'] / 10,
           alpha=0.25, s=10, zorder=100)
ax.set_xlim(0, 50)
ax.set_ylim(-26, 26)
ax.set_title('2024-25 Shot Attempts for Nikola Jokić')
plt.tight_layout()
plt.show()

In [ ]:
#| label: fig-jokic-25-zone
#| fig-cap: "Shot chart for the Denver Nuggets' center, Nikola Jokić. Here we specify different colors based on the zone in which the shot was taken."

fig, ax = plt.subplots(figsize=(10, 9))
court = NBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})
court.draw(ax=ax, display_range='offense')

zones = [z for z in jokic_shots['SHOT_ZONE_BASIC'].unique() if z != 'Backcourt']
colors = sns.color_palette('tab10', len(zones))

for zone, color in zip(zones, colors):
    zone_data = jokic_shots[jokic_shots['SHOT_ZONE_BASIC'] == zone]
    ax.scatter((420 - zone_data['LOC_Y']) / 10, zone_data['LOC_X'] / 10,
               alpha=0.25, s=10, color=color, label=zone, zorder=100)

ax.set_xlim(0, 50)
ax.set_ylim(-26, 26)
ax.legend(loc='upper right', fontsize=8)
ax.set_title('2024-25 Shot Attempts for Nikola Jokić by Zone')
plt.tight_layout()
plt.show()

### WNBA Shot Charts {.unnumbered}

The WNBA has seen tremendous growth in popularity and analytical interest in recent years. Just as we can create shot charts for NBA players, we can do the same for WNBA players. The `sportypy` module includes a `WNBACourt` class that draws a court with the correct WNBA dimensions, and the `isapy_data` module contains WNBA shot data for the 2025 season.

We start by loading the play-by-play shot data for the Seattle Storm during the 2025 WNBA season. The `wnba_seattle_pbp_2025` data set contains each shot's court coordinates and whether or not the shot was made. Since the data includes shots from both ends of the court, we filter to one half court for visualization.

In [ ]:
storm_pbp = isapy_data.load_data("wnba_seattle_pbp_2025")
storm_shots = storm_pbp[storm_pbp['coordinate_x'] < 0]

In [ ]:
#| label: fig-storm-25
#| fig-cap: "Shot chart for the Seattle Storm. Here we specify different colors based on either a make or a miss."

fig, ax = plt.subplots(figsize=(10, 9))
wnba_court = WNBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})
wnba_court.draw(ax=ax, display_range='defense')

colors = {True: '#2166AC', False: '#B2182B'}
for made, color in colors.items():
    data = storm_shots[storm_shots['scoring_play'] == made]
    ax.scatter(data['coordinate_x'], data['coordinate_y'],
               alpha=0.15, s=10, c=color, label=str(made), zorder=100)

ax.set_xlim(-50, 0)
ax.set_ylim(-26, 26)
ax.legend(title='Made Shot?')
ax.set_title('2025 Seattle Storm Shot Chart')
plt.tight_layout()
plt.show()

We can also compare individual player shot charts across the league. The `wnba_player_shots_2025` data set contains shot locations for Breanna Stewart and A'ja Wilson during the 2025 WNBA season. Stewart, a forward for the New York Liberty, and Wilson, a forward for the Las Vegas Aces, are two of the most dominant players in the league. Their shot charts reveal distinct offensive tendencies.

In [ ]:
#| label: fig-stewart-wilson
#| fig-cap: "Shot charts for Breanna Stewart and A'ja Wilson during the 2025 WNBA season."

wnba_shots = isapy_data.load_data("wnba_player_shots_2025")
wnba_shots = wnba_shots[
    (wnba_shots['coordinate_x'].abs() < 100) &
    (wnba_shots['coordinate_y'].abs() < 100) &
    (wnba_shots['coordinate_x'] < 0)
]

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

for ax, player in zip(axes, ['Breanna Stewart', "A'ja Wilson"]):
    wnba_court.draw(ax=ax, display_range='defense')
    player_data = wnba_shots[wnba_shots['athlete_display_name'] == player]
    ax.scatter(player_data['coordinate_x'], player_data['coordinate_y'],
               alpha=0.15, s=10, zorder=100)
    ax.set_xlim(-50, 0)
    ax.set_ylim(-26, 26)
    ax.set_title(player)

plt.suptitle('2025 WNBA Season Shot Charts')
plt.tight_layout()
plt.show()

___

## Lab Five: NBA Shots and Animation

One question that you might have after looking at several of the previous figures is "How can we look at these figures as an animation?" For example, you might be interested in looking at a particular player's shots year after year or compare individual players within the same season. Fortunately, `matplotlib.animation` makes turning a series of figures into an animation relatively easy.

![The core of the 2024-25 Denver Nuggets: (left to right) Jamal Murray, Nikola Jokić, Aaron Gordon, and Michael Porter Jr.](fig/nugs.jpeg){#fig-nugs-four}

Consider the four shot takers from the Denver Nuggets during the 2024-25 season: Jamal Murray, Nikola Jokić, Michael Porter Jr., and Aaron Gordon. Take the following code and see how it lays out the mechanics of how to create an animated shot chart for these players. For obvious reasons, this output does not appear in the printed edition of the text.

In [ ]:
#| eval: false

from matplotlib.animation import FuncAnimation

# Player IDs for Nuggets core
player_ids = {
    'Jamal Murray': 1627750,
    'Nikola Jokic': 203999,
    'Michael Porter Jr.': 1629008,
    'Aaron Gordon': 203932
}

# Fetch shots for each player
all_shots = {}
for name, pid in player_ids.items():
    time.sleep(1)
    sc = shotchartdetail.ShotChartDetail(
        team_id=den_team_id,
        player_id=pid,
        season_nullable='2024-25',
        season_type_all_star='Regular Season'
    )
    all_shots[name] = sc.get_data_frames()[0]

players_list = list(all_shots.keys())

court = NBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})

fig, ax = plt.subplots(figsize=(10, 9))

def animate(i):
    ax.clear()
    court.draw(ax=ax, display_range='offense')
    player = players_list[i]
    df = all_shots[player]
    backcourt = df[df['SHOT_ZONE_BASIC'] != 'Backcourt']
    ax.scatter((420 - backcourt['LOC_Y']) / 10, backcourt['LOC_X'] / 10,
               alpha=0.25, s=10, zorder=100)
    ax.set_xlim(0, 50)
    ax.set_ylim(-26, 26)
    ax.set_title(f'2024-25 Shots for Player: {player}')

anim = FuncAnimation(fig, animate, frames=len(players_list), interval=2000)
# Save as gif:
# anim.save('nuggets_shots.gif', writer='pillow')
plt.show()

1. Pick a player from the `nba_nuggets_shots_2025` data set and create an animation showing how the player's shots evolved over five or more seasons. What do you observe to have happened over the years?

In [ ]:
#| eval: false
#| include: false

nuggets_shots = isapy_data.load_data("nba_nuggets_shots_2025")

# Nikola Jokic has been with the Nuggets all 10 seasons
player_shots = nuggets_shots[
    (nuggets_shots['namePlayer'] == 'Nikola Jokic') &
    (nuggets_shots['zoneBasic'] != 'Backcourt')
]

court = NBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})

fig, ax = plt.subplots(figsize=(10, 9))
seasons = sorted(player_shots['yearSeason'].unique())

def animate_player(i):
    ax.clear()
    court.draw(ax=ax, display_range='offense')
    season = seasons[i]
    season_data = player_shots[player_shots['yearSeason'] == season]
    ax.scatter((420 - season_data['locationY']) / 10,
               season_data['locationX'] / 10,
               alpha=0.4, s=15, zorder=100)
    ax.set_xlim(0, 50)
    ax.set_ylim(-26, 26)
    ax.set_title(f'Nikola Jokic Shots: {season}')

anim = FuncAnimation(
    fig, animate_player, frames=len(seasons), interval=2000
)
# anim.save('jokic_by_season.gif', writer='pillow')
plt.show()

2. Construct an animated bar chart showing the count of each type of shot, whether or not it was made (`typeEvent`), over each season. The bars should represent each zone (`zoneBasic`) for each season.

3. Predict the probability of make/miss based on shot distance for Nikola Jokić during the 2024-25 season using a logistic regression model. Filter the data to use only shots less than or equal to 30 feet.

In [ ]:
#| eval: false
#| include: false

jokic_30 = jokic_shots[jokic_shots['SHOT_DISTANCE'] <= 30].copy()
jokic_30['made'] = jokic_30['SHOT_MADE_FLAG']
glm_shots = smf.logit('made ~ SHOT_DISTANCE', data=jokic_30).fit(disp=0)
print(glm_shots.summary())

___

## Case Study: Should We Trust The Process? Evaluating Talent in the NBA Before They Arrive

One of the side effects of the analytics revolution in sports has been the discovery of the value of draft picks. In general in major North American professional team sports, once per year professional teams get to select the rights to players, or "draft" them onto their teams from amateur or overseas professional leagues. There are of course many other side deals and negotiations that go on, but as the US/Canadian leagues are generally thought of as the top leagues in most team sports, with the top competition and the highest salaries (soccer and women's professional basketball are the most notable exceptions), there are no shortage of players wishing to come to play in those leagues.

![](fig/hinke.jpeg){width=50%}

The selection order for teams in the draft is in general inversely related to the order in which they finished the previous year. For example, in the 2023 NFL draft, the Chicago Bears were awarded the first pick (which they subsequently traded to Carolina), as at 3-14 they held the worst record amongst NFL teams in 2022-23. In that same NFL draft, the Kansas City Chiefs selected last, as they won the Super Bowl and were the champions of the 2022-23 NFL season. As the draft has seven rounds of selections, the teams select in that order seven times. So this meant the Bears were scheduled to select $1^{st}$, $33^{rd}$ (the first pick of the second round), $65^{th}$, $97^{th}$, and so on. This is a bit of a simplification as the NFL rules allow this to change a bit, but the point remains the same[^basketball-1].

[^basketball-1]: The leagues invented and refined this selection process to promote equality among the teams. That is, the worst teams are given the best chance to get better by getting to select new players first. This is just another way to promote equality in competition, much like salary caps and revenue sharing. It is interesting to the authors that these teams, whose owners have succeeded by and large in capitalism, here find it in their best interests to behave like socialists.

![](fig/embiid.jpeg){width=50%}

As the regular seasons in these sports come to a close, approximately 30-50% of teams (depending on the league) will qualify for the playoffs and a chance to win the championship. Teams who see that they will not make the playoffs may find it in their best interest to stop trying to win games in the current year in order to improve their position in the draft. This strategy is commonly known as tanking. One of the authors of this text discovered through research that as early as 25% of the way through the NBA season that playoff teams are already well determined (see the research block earlier in this paper). As clubs may try to improve their position more and more, they may begin to tank their games. The players themselves will not try to intentionally miss shots or lose games, as that would not be in their best interest in showing their value in the free agent market, but coaches and general managers may do subtle things to give themselves less of a chance of winning a game on any given night. This could include resting star players, not making optimal substitutions for situations, and not seeking more talented players to supplement the team in case of injuries.

This of course is a delicate balance. No league or team wants to be known as not trying to win, and fans who are still asked to pay premium dollars for tickets, parking, and beer become upset when the local team is sub-premium. This is true especially in basketball where one player can influence the team more than in any other sport as there are only five players on the court. Holding that good player out of the game can hurt the team, but the promises of getting that collegiate or foreign superstar onto your team the following year can be tempting. In fact, leagues have put in measures to prevent tanking, such as the NBA going to a weighted lottery process to determine the exact draft order, and in 2023-2024 have implemented new rules about how many games star players must play to qualify for statistical titles and for games on national TV.

Such is the story with the Philadelphia 76ers, and their analytics-minded general manager, Sam Hinkie. Hinkie took over the team in 2013, and the 76ers had not been good for quite a long time. In the previous year they missed the playoffs by one spot, and the two years before that they were amongst the last two teams to qualify for the playoffs. This put them in quite a perceived quandary: they were not good enough to really challenge for a championship, and yet they were not bad enough to get one of the best two or three players available in the next draft. Hinkie used a strategy to trade current players to another team in return for the use of their spot in the next draft. He did not view this as tanking, but prioritizing wins in the long term over those in the short term. He informed people it would get worse before it got better, but that they needed to "Trust the Process".

In the 2013-2014 season, the 76ers finished $29^{th}$ out of 30 teams in the NBA, ahead of only the 2021 NBA champion Milwaukee Bucks. In 2014-2015 they had a virtually identical record, finishing one game ahead of the New York Knicks and two games ahead of the Minnesota Timberwolves. In 2015-2016 they finished in last place, winning only 10 out of 82 games and nearly matching the record for worst NBA team ever (the 9-73 76ers of 1972-1973). With each loss, "Trust The Process" became the rallying cry, a cry that was getting harder and harder to believe with years of misery and barely remembering their last trip to the NBA finals in 2001. Before the season ended, Hinkie resigned from the team.

Was the plug pulled on Hinkie and The Process too soon? In 2016-2017 they won 17 more games than the previous year, but still finished $27^{th}$ out of 30 teams in the league. In the following two years, the team finished 52-30 and 51-31 and returned to the playoffs for the first time since 2011-2012. They had a similar winning percentage in the COVID-19 interrupted season of 2019-2020 at 43-30, and after that season hired Hinkie's mentor, Daryl Morey, as the president of basketball operations. In the COVID-19 shortened 2020-2021 season, the 76ers finished first in the Eastern Conference and 3rd overall with a 49-23 record but lost to Atlanta in the 2nd round of the playoffs. 2021-2022 and 2022-2023 saw similar results: 2nd-place finishes in the Atlantic Division and losses in the 2nd round of the playoffs.

This has happened in other sports as well -- a stockpiling of draft picks and young talent who will likely lose currently but in the future are believed to have a good chance of winning. Recall from the research note in the baseball chapter on the strategy the Astros used to build a championship team. Indeed, they won the World Series in 2017, though not without the other controversy mentioned in that research note.

1. How would you use analytics to verify if this is a good long-term strategy?
2. What are the issues with using data to evaluate players that are not currently playing in the league/level that top-flight league administrators aspire for them to compete?
3. How do you communicate this to the fanbase?

___

## Exercises

Use datasets provided in the `isapy_data` module to answer the following questions.

1. Create a new column in the `nba_games_2025` table for Free Throw Rate (FTR). You will need to use the `fga` and `fta` columns in the table. Plot your new variable against `fta_rate` from the `nba_ff_team_2025` table.

In [ ]:
#| eval: false
#| include: false

games = isapy_data.load_data("nba_games_2025")
ff = isapy_data.load_data("nba_ff_team_2025")
df_ff_games = games.merge(ff, on=['game_id', 'team_id'], suffixes=('', '_ff'))
df_ff_games['ftr'] = df_ff_games['fta'] / df_ff_games['fga']

plt.figure(figsize=(8, 6))
plt.scatter(df_ff_games['ftr'], df_ff_games['fta_rate'], alpha=0.5)
plt.xlabel('FTR (calculated)')
plt.ylabel('FTA Rate (NBA.com)')
plt.title('Free Throw Attempt Rate')
plt.tight_layout()
plt.show()

2. The NBA defines pace as the number of possessions per 48 minutes for a team or a player, and is denoted as pace in the advanced statistics table. Compute the average pace for the Denver Nuggets when they play at home and when they play away from Denver. The data are in the `nba_adv_team_2025` table and you will need to join this with the `game_summary` object defined in the code block below.

In [ ]:
#| eval: false

game_summary = games.copy()
game_summary['location'] = game_summary['matchup'].apply(
    lambda x: 'away' if '@' in x else 'home'
)
game_summary = game_summary[['game_date', 'game_id', 'team_id',
                              'team_abbreviation', 'matchup', 'location',
                              'wl', 'plus_minus']]

In [ ]:
#| eval: false
#| include: false

adv = isapy_data.load_data("nba_adv_team_2025")
tmp = game_summary.merge(adv, on=['game_id', 'team_id'])
result = (tmp.groupby(['team_name', 'location'])['pace']
          .mean()
          .reset_index(name='avg_pace')
          .sort_values('avg_pace', ascending=False))
print(result[result['team_name'].str.contains('Nuggets')])

3. Construct a new DataFrame (call it `df_ff`) that has `game_id`, `game_date`, `team_name`, the four factors (from `nba_ff_team_2025`), offensive rating (`off_rating`), defensive rating (`def_rating`), win/loss (`wl`), location (`location`), and Plus/Minus (`plus_minus`). Create an `is_win` variable to be `True` if it was a win and `False` if a loss.

In [ ]:
#| eval: false
#| include: false

adv = isapy_data.load_data("nba_adv_team_2025")
ff = isapy_data.load_data("nba_ff_team_2025")

df_ff = adv.merge(ff, on=['game_id', 'team_id', 'team_name',
                           'team_abbreviation', 'team_city'],
                  suffixes=('_adv', '_ff'))
df_ff = df_ff.merge(game_summary, on=['game_id', 'team_id'])
df_ff = df_ff.rename(columns={'efg_pct_ff': 'efg_pct',
                                'tm_tov_pct_ff': 'tm_tov_pct',
                                'oreb_pct_ff': 'oreb_pct'})
df_ff = df_ff[['game_date', 'game_id', 'team_name', 'oreb_pct', 'tm_tov_pct',
               'efg_pct', 'fta_rate', 'off_rating', 'def_rating', 'wl',
               'location', 'plus_minus']].copy()
df_ff['is_win'] = df_ff['wl'] == 'W'
print(df_ff[['efg_pct', 'oreb_pct', 'tm_tov_pct', 'fta_rate']].corr())

4. Using the `nba_games_2025` and `nba_adv_team_2025` data sets to obtain the advanced statistics for both teams in a game between the Denver Nuggets and the Dallas Mavericks during the 2024-25 season. Compute the z scores for offensive and defensive ratings for both teams relative to the league distributions.

In [ ]:
#| eval: false
#| include: false

games = isapy_data.load_data("nba_games_2025")
adv = isapy_data.load_data("nba_adv_team_2025")

game_summary = games.copy()
game_summary['location'] = game_summary['matchup'].apply(
    lambda x: 'away' if '@' in x else 'home'
)

den_dal = games[
    (games['team_abbreviation'] == 'DEN') &
    (games['matchup'].str.contains('DAL'))
]
den_dal_game_id = den_dal['game_id'].iloc[0]
den_dal_adv = adv[adv['game_id'] == den_dal_game_id]

z_off = (den_dal_adv['off_rating'].values - adv['off_rating'].mean()) / adv['off_rating'].std()
z_def = (den_dal_adv['def_rating'].values - adv['def_rating'].mean()) / adv['def_rating'].std()
print("Offensive rating Z-scores:", z_off)
print("Defensive rating Z-scores:", z_def)
print("Offensive rating percentiles:", norm.cdf(z_off))
print("Defensive rating percentiles:", norm.cdf(z_def))

5. Using the `df_ff` data set, run a multiple regression model using the four factors and offensive rating to predict `plus_minus`. What are the predicted values of `plus_minus` for both teams from Problem 4?

In [ ]:
#| eval: false
#| include: false

ff_lm = smf.ols('plus_minus ~ oreb_pct + tm_tov_pct + efg_pct + fta_rate + off_rating',
                  data=df_ff).fit()
print(ff_lm.summary())

6. Use the data related to the Nuggets shots from 2015 through 2025 and create a bar chart showing the total number of observations by season. Show the zone (`zoneBasic`) as a color within each bar. Filter out backcourt shots.

In [ ]:
#| eval: false
#| include: false

nuggets_shots = isapy_data.load_data("nba_nuggets_shots_2025")
shots_no_bc = nuggets_shots[nuggets_shots['zoneBasic'] != 'Backcourt']

shot_counts = shots_no_bc.groupby(['yearSeason', 'zoneBasic']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
shot_counts.plot(kind='bar', stacked=True, ax=ax, colormap='Set1')
ax.set_title('Denver Nuggets Shot Tendencies, 2015-2025')
ax.set_xlabel('Season')
ax.set_ylabel('Number of Shots')
plt.legend(title='Zone', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

7. Who took the most shots in each season for the Nuggets?

In [ ]:
#| eval: false
#| include: false

nuggets_shots = isapy_data.load_data("nba_nuggets_shots_2025")
result = (nuggets_shots.groupby(['slugSeason', 'namePlayer'])
          .size()
          .reset_index(name='n')
          .sort_values(['slugSeason', 'n'], ascending=[True, False])
          .groupby('slugSeason')
          .first()
          .reset_index())
print(result[['slugSeason', 'namePlayer', 'n']])

8. Compute and plot the shot chart for Denver's 2024-25 season. Use points colored by the zone.

In [ ]:
#| eval: false
#| include: false

nuggets_shots = isapy_data.load_data("nba_nuggets_shots_2025")
season_shots = nuggets_shots[
    (nuggets_shots['yearSeason'] == 2025) &
    (nuggets_shots['zoneBasic'] != 'Backcourt')
]

fig, ax = plt.subplots(figsize=(10, 9))
court = NBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})
court.draw(ax=ax, display_range='offense')

zones = season_shots['zoneBasic'].unique()
colors = sns.color_palette('tab10', len(zones))
for zone, color in zip(zones, colors):
    zone_data = season_shots[season_shots['zoneBasic'] == zone]
    ax.scatter((420 - zone_data['locationY']) / 10,
               zone_data['locationX'] / 10,
               alpha=0.25, s=10, color=color, label=zone, zorder=100)

ax.set_xlim(0, 50)
ax.set_ylim(-26, 26)
ax.legend(loc='upper right', fontsize=8)
ax.set_title('Denver Nuggets Shot Attempts, 2024-25')
plt.tight_layout()
plt.show()

9. Add a layer to the previous figure showing points shaped by `typeEvent` (make or miss the shot).

In [ ]:
#| eval: false
#| include: false

fig, ax = plt.subplots(figsize=(10, 9))
court = NBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})
court.draw(ax=ax, display_range='offense')

markers = {'Made Shot': 'o', 'Missed Shot': 'x'}
zones = season_shots['zoneBasic'].unique()
colors = sns.color_palette('tab10', len(zones))
zone_color_map = dict(zip(zones, colors))

for event_type, marker in markers.items():
    event_data = season_shots[season_shots['typeEvent'] == event_type]
    for zone in zones:
        zone_data = event_data[event_data['zoneBasic'] == zone]
        ax.scatter((420 - zone_data['locationY']) / 10,
                   zone_data['locationX'] / 10,
                   alpha=0.15, s=10,
                   color=zone_color_map[zone],
                   marker=marker, zorder=100)

ax.set_xlim(0, 50)
ax.set_ylim(-26, 26)
ax.set_title('Denver Nuggets Shot Chart, 2024-25')
plt.tight_layout()
plt.show()

10. Compare Michael Porter Jr.'s shots from the 2023-24 and 2024-25 seasons. Create a 2x2 facet plot showing his makes/misses by zone for both years.

In [ ]:
#| eval: false
#| include: false

nuggets_shots = isapy_data.load_data("nba_nuggets_shots_2025")
mpj = nuggets_shots[
    (nuggets_shots['namePlayer'] == 'Michael Porter Jr.') &
    (nuggets_shots['yearSeason'].isin([2024, 2025])) &
    (nuggets_shots['zoneBasic'] != 'Backcourt')
]

court = NBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for i, event in enumerate(['Made Shot', 'Missed Shot']):
    for j, season in enumerate([2024, 2025]):
        ax = axes[i][j]
        court.draw(ax=ax, display_range='offense')
        data = mpj[(mpj['typeEvent'] == event) & (mpj['yearSeason'] == season)]
        zones = data['zoneBasic'].unique()
        colors = sns.color_palette('tab10', len(zones))
        for zone, color in zip(zones, colors):
            zone_data = data[data['zoneBasic'] == zone]
            ax.scatter((420 - zone_data['locationY']) / 10,
                       zone_data['locationX'] / 10,
                       alpha=0.25, s=10, color=color, label=zone, zorder=100)
        ax.set_xlim(0, 50)
        ax.set_ylim(-26, 26)
        ax.set_title(f'{event} - {season}')

plt.suptitle('Michael Porter Jr. Shot Charts')
plt.tight_layout()
plt.show()

11. Download the shot charts for the 2024-25 Golden State Warriors and construct a heat map showing the probability of Stephen Curry making shots.

In [ ]:
#| eval: false
#| include: false

from scipy.ndimage import gaussian_filter

curry_id = 201939
gsw_team_id = 1610612744

time.sleep(1)
sc = shotchartdetail.ShotChartDetail(
    team_id=gsw_team_id,
    player_id=curry_id,
    season_nullable='2024-25',
    season_type_all_star='Regular Season'
)
steph_df = sc.get_data_frames()[0]

# Transform coordinates to sportypy system
steph_df['court_x'] = (420 - steph_df['LOC_Y']) / 10
steph_df['court_y'] = steph_df['LOC_X'] / 10

# Create grid in sportypy coordinates
x_bins = np.linspace(0, 50, 50)
y_bins = np.linspace(-26, 26, 50)

# Bin all shots and made shots
all_hist, xedges, yedges = np.histogram2d(steph_df['court_x'], steph_df['court_y'],
                                           bins=[x_bins, y_bins])
made = steph_df[steph_df['SHOT_MADE_FLAG'] == 1]
made_hist, _, _ = np.histogram2d(made['court_x'], made['court_y'],
                                  bins=[x_bins, y_bins])

# Smooth and compute probability
all_smooth = gaussian_filter(all_hist, sigma=2)
made_smooth = gaussian_filter(made_hist, sigma=2)
prob = np.divide(made_smooth, all_smooth, where=all_smooth > 0,
                 out=np.zeros_like(all_smooth))

fig, ax = plt.subplots(figsize=(10, 9))
court = NBACourt(color_updates={
    'plot_background': '#ffffff',
    'offensive_half_court': '#ffffff',
    'defensive_half_court': '#ffffff',
    'court_apron': '#ffffff',
    'two_point_range': '#ffffff',
    'painted_area': '#ffffff',
    'center_circle_fill': '#ffffff',
    'free_throw_circle_fill': '#ffffff'
})
court.draw(ax=ax, display_range='offense')

x_centers = (xedges[:-1] + xedges[1:]) / 2
y_centers = (yedges[:-1] + yedges[1:]) / 2
X, Y = np.meshgrid(x_centers, y_centers)

im = ax.pcolormesh(X, Y, prob.T, cmap='Spectral', alpha=0.75,
                    shading='auto', vmin=0, vmax=0.7, zorder=99)
plt.colorbar(im, ax=ax, label='P(Make)')
ax.set_xlim(0, 50)
ax.set_ylim(-26, 26)
ax.set_title('Stephen Curry 2024-25 Made Shot Probabilities')
plt.tight_layout()
plt.show()